In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from datetime import date, timedelta, datetime

from ortools.sat.python import cp_model

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# Load Models
solar_model = joblib.load('../models/solar_forecaster_v1.joblib')
demand_model = joblib.load('../models/demand_forecaster_v1.joblib')

# Load historical data for demand lag feature
historical_df = pd.read_csv('../data/hybrid_park_dataset.csv', parse_dates=['Timestamp'], index_col='Timestamp')

✓ Models and historical data loaded.


In [ ]:
def create_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates time-series features from a datetime index."""
    df['hour'] = df['Timestamp'].dt.hour
    df['day_of_year'] = df['Timestamp'].dt.dayofyear
    df['day_of_week'] = df['Timestamp'].dt.dayofweek
    df['is_weekend'] = (df['Timestamp'].dt.dayofweek >= 5).astype(int)
    df['month'] = df['Timestamp'].dt.month
    return df

def create_cyclical_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates cyclical features for time-based attributes."""
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.0)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.0)
    return df

SOLAR_FEATURES = ['Temperature_C', 'Cloud_Cover_Pct', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos']
DEMAND_FEATURES = ['Temperature_C', 'Demand_Lag_24h', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'day_of_week', 'is_weekend']

def generate_solar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = create_time_features(df)
    df = create_cyclical_features(df)
    day_of_year = df['day_of_year']
    hour = df['hour']
    seasonal_avg = 25.5 + 17.5 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
    daily_offset = 12 * np.sin(2 * np.pi * (hour - 4) / 24)
    df['Temperature_C'] = seasonal_avg + daily_offset
    base_cloud = 0.05 + 0.1 * np.sin(np.pi * day_of_year / 365)
    is_monsoon = df['Timestamp'].dt.month.isin([7, 8])
    monsoon_effect = 0.5 * (day_of_year % 14) / 14
    is_spike_day = (day_of_year % 20 == 0)
    spike_effect = 0.6
    df['Cloud_Cover_Pct'] = np.where(is_monsoon, base_cloud + monsoon_effect, np.where(is_spike_day, base_cloud + spike_effect, base_cloud))
    df['Cloud_Cover_Pct'] = df['Cloud_Cover_Pct'].clip(0, 1)
    return df

def generate_demand_features(df: pd.DataFrame, previous_day_demand: list) -> pd.DataFrame:
    df = create_time_features(df)
    df = create_cyclical_features(df)
    day_of_year = df['day_of_year']
    hour = df['hour']
    seasonal_avg = 25.5 + 17.5 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
    daily_offset = 12 * np.sin(2 * np.pi * (hour - 4) / 24)
    df['Temperature_C'] = seasonal_avg + daily_offset
    df['Demand_Lag_24h'] = previous_day_demand
    return df

def get_forecasts(prediction_date: date):
    """Generates solar and demand forecasts for a given date."""
    # Solar Forecast
    timestamps = pd.date_range(start=prediction_date, periods=24, freq='h')
    solar_df = pd.DataFrame(timestamps, columns=['Timestamp'])
    solar_features = generate_solar_features(solar_df)
    solar_predictions = solar_model.predict(solar_features[SOLAR_FEATURES]).clip(0)

    # Demand Forecast
    previous_year_date = prediction_date.replace(year=prediction_date.year - 1)
    previous_day_demand_series = historical_df.loc[historical_df.index.date == previous_year_date, 'Demand_MW']
    if len(previous_day_demand_series) < 24:
        raise ValueError(f"Missing historical data for {previous_year_date}")
    previous_day_demand = previous_day_demand_series.head(24).tolist()
    demand_df = pd.DataFrame(timestamps, columns=['Timestamp'])
    demand_features = generate_demand_features(demand_df, previous_day_demand)
    demand_predictions = demand_model.predict(demand_features[DEMAND_FEATURES]).clip(0)

    return solar_predictions.tolist(), demand_predictions.tolist()

✓ Forecasting functions defined.


In [ ]:
def solve_economic_dispatch(demand_forecast, solar_forecast, coal_min_mw, coal_max_mw, ramp_rate_mw_per_step):
    """
    Solves the economic dispatch problem for a hybrid power park.
    
    Args:
        demand_forecast (list): 96-step (15-min) demand forecast in MW.
        solar_forecast (list): 96-step (15-min) solar forecast in MW.
        coal_min_mw (int): Minimum technical limit for the coal plant.
        coal_max_mw (int): Maximum capacity of the coal plant.
        ramp_rate_mw_per_step (int): Max MW change for coal plant per 15-min step.
    
    Returns:
        A tuple containing (status, results_df).
    """
    model = cp_model.CpModel()
    
    num_steps = len(demand_forecast)
    

    solar_dispatch = [model.NewIntVar(0, int(solar_forecast[i]), f'solar_{i}') for i in range(num_steps)]
    coal_dispatch = [model.NewIntVar(coal_min_mw, coal_max_mw, f'coal_{i}') for i in range(num_steps)]
    total_generation = [model.NewIntVar(0, coal_max_mw + int(max(solar_forecast)), f'total_{i}') for i in range(num_steps)]
    
    shortage = [model.NewIntVar(0, 2000, f'shortage_{i}') for i in range(num_steps)] 
    over_generation = [model.NewIntVar(0, 2000, f'over_gen_{i}') for i in range(num_steps)] 

    for i in range(num_steps):
        model.Add(total_generation[i] == solar_dispatch[i] + coal_dispatch[i])
        model.Add(total_generation[i] + shortage[i] == int(demand_forecast[i]) + over_generation[i])

    for i in range(1, num_steps):
        model.Add(coal_dispatch[i] - coal_dispatch[i-1] <= ramp_rate_mw_per_step) 
        model.Add(coal_dispatch[i-1] - coal_dispatch[i] <= ramp_rate_mw_per_step)

    total_coal_cost = sum(coal_dispatch)
    
    total_shortage_penalty = sum(shortage) * 1000 
    total_over_gen_penalty = sum(over_generation) * 100 
    
    model.Minimize(total_coal_cost + total_shortage_penalty + total_over_gen_penalty)


    solver = cp_model.CpSolver()
    solver.parameters.log_search_progress = False
    status = solver.Solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        results = {
            'demand_mw': demand_forecast,
            'solar_mw': solar_forecast,
            'solar_used_mw': [solver.Value(s) for s in solar_dispatch],
            'coal_mw': [solver.Value(c) for c in coal_dispatch],
            'total_mw': [solver.Value(t) for t in total_generation],
            'shortage_mw': [solver.Value(s) for s in shortage],
            'overgen_mw': [solver.Value(o) for o in over_generation]
        }
        results_df = pd.DataFrame(results)
        results_df['curtailment_mw'] = results_df['solar_mw'] - results_df['solar_used_mw']
        return solver.StatusName(status), results_df
    else:
        return solver.StatusName(status), None

✓ OR-Tools solver function defined.


In [ ]:
CO2_TONS_PER_MWH_COAL = 0.98
COST_INR_PER_MWH_COAL = 4000.0

def _generate_alerts_and_table(df_15min):
    alerts = []
    table = []
    if df_15min['shortage_mw'].sum() > 0:
        first_shortage = df_15min[df_15min['shortage_mw'] > 0].iloc[0]
        alerts.append({
            "time": first_shortage['time'],
            "action": "ALERT_SHORTAGE",
            "value_mw": round(first_shortage['shortage_mw'], 2),
            "reason": "Demand exceeds physical ramp/capacity limits. Load shedding required."
        })
    
    table.append({
        "time": "05:00",
        "action": "RAMP_UP",
        "value_mw": 20.0,
        "reason": "Demand increasing"
    })
    return alerts, table

def _calculate_summary_metrics(df_15min, mwh_factor):
    summary = {}
    summary['total_demand_mwh'] = df_15min['demand_mw'].sum() * mwh_factor
    summary['total_solar_available_mwh'] = df_15min['solar_mw'].sum() * mwh_factor
    summary['total_solar_used_mwh'] = df_15min['solar_used_mw'].sum() * mwh_factor
    summary['total_coal_dispatch_mwh'] = df_15min['coal_mw'].sum() * mwh_factor
    summary['total_curtailed_mwh'] = df_15min['curtailment_mw'].sum() * mwh_factor
    summary['total_shortage_mwh'] = df_15min['shortage_mw'].sum() * mwh_factor
    summary['total_overgen_mwh'] = df_15min['overgen_mw'].sum() * mwh_factor

    baseline_coal_mwh = summary['total_demand_mwh']

    summary['coal_saved_mwh'] = baseline_coal_mwh - summary['total_coal_dispatch_mwh']
    summary['co2_avoided_tons'] = summary['coal_saved_mwh'] * CO2_TONS_PER_MWH_COAL
    summary['cost_savings_inr'] = summary['coal_saved_mwh'] * COST_INR_PER_MWH_COAL

    if baseline_coal_mwh > 0:
        summary['coal_reduction_percent'] = (summary['coal_saved_mwh'] / baseline_coal_mwh) * 100
    else:
        summary['coal_reduction_percent'] = 0

    if summary['total_solar_available_mwh'] > 0:
        summary['solar_utilization_percent'] = (summary['total_solar_used_mwh'] / summary['total_solar_available_mwh']) * 100
    else:
        summary['solar_utilization_percent'] = 0
        
    return {k: round(v, 2) for k, v in summary.items()}, baseline_coal_mwh

def create_dispatch_schedule(prediction_date, hourly_solar, hourly_demand):
    COAL_MIN_MW = 400
    COAL_MAX_MW = 800
    COAL_RAMP_RATE_MW_PER_HOUR = 80
    TIME_STEP_MINUTES = 15
    
    steps_per_hour = 60 // TIME_STEP_MINUTES
    demand_forecast_15min = np.repeat(hourly_demand, steps_per_hour)
    solar_forecast_15min = np.repeat(hourly_solar, steps_per_hour)
    
    ramp_rate_per_step = COAL_RAMP_RATE_MW_PER_HOUR // steps_per_hour
    status, results_df_15min = solve_economic_dispatch(
        demand_forecast_15min.tolist(), solar_forecast_15min.tolist(),
        COAL_MIN_MW, COAL_MAX_MW, ramp_rate_per_step
    )

    if results_df_15min is None:
        return None, f"Optimizer failed: {status}"

    mwh_factor = TIME_STEP_MINUTES / 60
    timestamps_15min = pd.to_datetime(pd.date_range(start=prediction_date, periods=len(results_df_15min), freq=f'{TIME_STEP_MINUTES}min'))
    results_df_15min['timestamp'] = timestamps_15min
    results_df_15min['time'] = results_df_15min['timestamp'].dt.strftime('%H:%M')

    hourly_data_df = results_df_15min.resample('h', on='timestamp').mean(numeric_only=True)
    hourly_timestamps = hourly_data_df.index
    hourly_data_df['timestamp'] = hourly_timestamps.map(lambda x: x.isoformat())
    hourly_data_df['time'] = hourly_timestamps.strftime('%H:%M')
    
    summary_metrics, baseline_coal_mwh = _calculate_summary_metrics(results_df_15min, mwh_factor)
    alerts, table = _generate_alerts_and_table(results_df_15min)
    peak_solar_row = results_df_15min.loc[results_df_15min['solar_mw'].idxmax()]
    peak_solar = {"solar_mw": round(peak_solar_row['solar_mw'], 2), "time": peak_solar_row['time']}
    
    # Simplified for notebook output
    output_json = {"summary": summary_metrics, "data": hourly_data_df.round(2).to_dict(orient='records')}
    return output_json, None

In [ ]:
PREDICTION_DATE = date(2026, 4, 13)
COAL_MIN_MW = 400
COAL_MAX_MW = 800
COAL_RAMP_RATE_MW_PER_HOUR = 80
TIME_STEP_MINUTES = 15

print(f"▶ Generating forecasts for {PREDICTION_DATE}...")
hourly_solar, hourly_demand = get_forecasts(PREDICTION_DATE)

steps_per_hour = 60 // TIME_STEP_MINUTES
demand_forecast_15min = np.repeat(hourly_demand, steps_per_hour)
solar_forecast_15min = np.repeat(hourly_solar, steps_per_hour)
print(f"✓ Upsampled forecasts from 24 hourly steps to {len(demand_forecast_15min)} {TIME_STEP_MINUTES}-minute steps.")

ramp_rate_per_step = COAL_RAMP_RATE_MW_PER_HOUR // steps_per_hour
print(f"▶ Running optimizer with Coal Min: {COAL_MIN_MW} MW, Max: {COAL_MAX_MW} MW, Ramp: {ramp_rate_per_step} MW/{TIME_STEP_MINUTES}min...")
status, results_df = solve_economic_dispatch(
    demand_forecast_15min, 
    solar_forecast_15min, 
    COAL_MIN_MW, 
    COAL_MAX_MW, 
    ramp_rate_per_step
)
print(f"✓ Optimizer finished with status: {status}")


if results_df is not None:
    timestamps_15min = pd.date_range(start=PREDICTION_DATE, periods=len(results_df), freq=f'{TIME_STEP_MINUTES}min')
    results_df['timestamp'] = timestamps_15min
    results_df['time'] = results_df['timestamp'].dt.strftime('%H:%M')

    mwh_factor = TIME_STEP_MINUTES / 60
    total_demand_mwh = results_df['demand_mw'].sum() * mwh_factor
    total_solar_mwh = results_df['solar_mw'].sum() * mwh_factor
    total_coal_mwh = results_df['coal_mw'].sum() * mwh_factor
    total_curtailed_mwh = results_df['curtailment_mw'].sum() * mwh_factor
    total_shortage_mwh = results_df['shortage_mw'].sum() * mwh_factor
    total_overgen_mwh = results_df['overgen_mw'].sum() * mwh_factor
    
    results_df['timestamp'] = results_df['timestamp'].apply(lambda x: x.isoformat())

    output_json = {
        "meta": {
            "time_step_minutes": TIME_STEP_MINUTES,
            "horizon_hours": 24,
            "generated_at": datetime.now().isoformat(),
            "status": status
        },
        "summary": {
            "total_demand_mwh": round(total_demand_mwh, 2),
            "total_solar_mwh": round(total_solar_mwh, 2),
            "total_coal_mwh": round(total_coal_mwh, 2),
            "total_curtailed_mwh": round(total_curtailed_mwh, 2),
            "total_shortage_mwh": round(total_shortage_mwh, 2),
            "total_overgen_mwh": round(total_overgen_mwh, 2),
        },
        "data": results_df.round(2).to_dict(orient='records')
    }

    output_path = '../optimizer/dispatch_output.json'
    with open(output_path, 'w') as f:
        json.dump(output_json, f, indent=2)
    
    print(f"✓ Dispatch schedule saved to {output_path}")
    
    display(results_df.head())

▶ Generating forecasts for 2026-04-13...
✓ Upsampled forecasts from 24 hourly steps to 96 15-minute steps.
▶ Running optimizer with Coal Min: 400 MW, Max: 800 MW, Ramp: 20 MW/15min...
✓ Optimizer finished with status: OPTIMAL
✓ Dispatch schedule saved to ../optimizer/dispatch_output.json


,demand_mw,solar_mw,solar_used_mw,coal_mw,total_mw,shortage_mw,overgen_mw,curtailment_mw,timestamp,time
0,1003.383850,0.0,0,800,800,203,0,0.0,2026-04-13T00:00:00,00:00
1,1003.383850,0.0,0,800,800,203,0,0.0,2026-04-13T00:15:00,00:15
2,1003.383850,0.0,0,800,800,203,0,0.0,2026-04-13T00:30:00,00:30
3,1003.383850,0.0,0,800,800,203,0,0.0,2026-04-13T00:45:00,00:45
4,1003.830688,0.0,0,800,800,203,0,0.0,2026-04-13T01:00:00,01:00
